In [26]:
from pathlib import Path
import json
import warnings

import geopandas as gpd
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import statsmodels.formula.api as smf
from linearmodels.panel import PanelOLS
from matplotlib.ticker import MaxNLocator

warnings.filterwarnings("ignore", category=FutureWarning)

DATA_FILE = Path("nfhs_full5-30.csv")
GEO_FILE = Path("data/india_states.geojson")
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

# Publication-style defaults
plt.rcParams.update({
    "figure.dpi": 120,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 10,
    "axes.titlesize": 11,
    "axes.labelsize": 10,
    "legend.fontsize": 9,
})

df_nfhs = pd.read_csv(DATA_FILE)
df_nfhs.head()

,state,round,year,area,section,indicator,harmonized_indicator,value,data_flag,source
0,India,NFHS-6,2023-24,Urban,Population and Household Profile,Population below age 5 years (%),NaN,6.6,NaN,NFHS-6 PDF
1,India,NFHS-6,2023-24,Rural,Population and Household Profile,Population below age 5 years (%),NaN,8.6,NaN,NFHS-6 PDF
2,India,NFHS-6,2023-24,Total,Population and Household Profile,Population below age 5 years (%),NaN,8.0,NaN,NFHS-6 PDF
3,India,NFHS-6,2023-24,Urban,Population and Household Profile,Population below age 15 years (%),Population below age 15 years (%),22.0,NaN,NFHS-6 PDF
4,India,NFHS-6,2023-24,Rural,Population and Household Profile,Population below age 15 years (%),Population below age 15 years (%),27.0,NaN,NFHS-6 PDF


In [20]:
FERTILITY_KW   = ["fertility rate", "tfr", "total fertility",
                        "birth rate", "children born", "births per"]
PHONE_KW       = ["mobile phone", "mobile telephone", "cell phone",
                        "smartphone", "phone ownership", "use mobile"]
INTERNET_KW    = ["internet", "online"]
EDUCATION_KW   = ["literate", "literacy", "education", "schooling",
                        "no education", "secondary", "years of school"]
WEALTH_KW      = ["wealth", "poorest", "richest", "quintile",
                        "household assets", "below poverty"]
URBAN_KW       = ["urban", "rural"]
CONTRACEPTION_KW     = ["contracepti", "family planning", "modern method",
                        "unmet need", "steriliz"]
EMPOWERMENT_KW       = ["decision", "autonomy", "empowerment",
                        "own money", "bank account", "financial"]

## Phase 1 — Indicator mapping & panel construction

Harmonised indicators from Pakhare & Joshi (NFHS India Explorer). Analysis unit: **state/UT × NFHS round**, `area = Total`.

In [ ]:
# Exact harmonised-indicator strings (cross-round comparable where flagged in meta.json)
INDICATORS = {
    "tfr": "Total fertility rate (children per woman)",
    "mobile": "Women with own mobile phone (%)",
    "internet": "Women who ever used the internet (%)",
    "literacy": "Women (15-49) who are literate (%)",
    "schooling": "Women (15-49) with 10+ years schooling (%)",
    "fp_modern": "FP: Any modern method (%)",
    "unmet_fp": "Total unmet need for FP (%)",
    "hh_decisions": "Married women participate in HH decisions (%)",
    "electricity": "Households with electricity (%)",
    "bank_account": "Women with own bank/savings account (%)",
}
INV_INDICATORS = {v: k for k, v in INDICATORS.items()}

WAVE_MAP = {"NFHS-3": 3, "NFHS-4": 4, "NFHS-5": 5, "NFHS-6": 6}
YEAR_MID = {"NFHS-3": 2005.5, "NFHS-4": 2015.5, "NFHS-5": 2020.0, "NFHS-6": 2023.5}

MAIN_ROUNDS = ["NFHS-4", "NFHS-5", "NFHS-6"]
INTERNET_ROUNDS = ["NFHS-5", "NFHS-6"]
EXCLUDE_STATES = {"India"}


def build_panel(df, rounds, indicators=INDICATORS, area="Total"):
    """Long → wide state×round panel for selected harmonised indicators."""
    target_inds = set(indicators.values())
    sub = df[
        (df["area"] == area)
        & (~df["state"].isin(EXCLUDE_STATES))
        & (df["round"].isin(rounds))
        & (df["harmonized_indicator"].isin(target_inds))
    ].copy()

    sub["_flag"] = sub["data_flag"].notna().astype(int)
    sub = sub.sort_values("_flag").drop_duplicates(
        ["state", "round", "harmonized_indicator"], keep="first"
    )

    wide = sub.pivot(index=["state", "round"], columns="harmonized_indicator", values="value")
    wide = wide.rename(columns=INV_INDICATORS)
    wide = wide.reset_index()
    wide["wave"] = wide["round"].map(WAVE_MAP)
    wide["year_mid"] = wide["round"].map(YEAR_MID)
    return wide


panel_all = build_panel(df_nfhs, rounds=list(WAVE_MAP.keys()))
panel_main = build_panel(df_nfhs, rounds=MAIN_ROUNDS)

required = ["tfr", "mobile"]
panel_est = panel_main.dropna(subset=required).copy()
panel_est = panel_est.set_index(["state", "wave"]).sort_index()

panel_internet = build_panel(df_nfhs, rounds=INTERNET_ROUNDS)
panel_internet_est = panel_internet.dropna(subset=["tfr", "internet"]).copy()
panel_internet_est = panel_internet_est.set_index(["state", "wave"]).sort_index()

panel_main.to_csv(OUT_DIR / "panel_state_total.csv", index=False)
print(f"Panel (NFHS-4/5/6): {panel_main.shape[0]} state-rounds, {panel_main['state'].nunique()} states")
print(f"Estimation sample (complete TFR + mobile): {panel_est.shape[0]} obs, "
      f"{panel_est.index.get_level_values(0).nunique()} states")
panel_est.head()

### Diagnostics — coverage, attrition, unmapped phone indicators

In [ ]:
def coverage_matrix(df_long, harmonized_name, area="Total"):
    sub = df_long[
        (df_long["area"] == area)
        & (~df_long["state"].isin(EXCLUDE_STATES))
        & (df_long["harmonized_indicator"] == harmonized_name)
    ]
    return sub.pivot_table(index="state", columns="round", values="value", aggfunc="first")

cov_tfr = coverage_matrix(df_nfhs, INDICATORS["tfr"])
cov_mobile = coverage_matrix(df_nfhs, INDICATORS["mobile"])

print("=== TFR coverage (non-null state-rounds) ===")
print(cov_tfr.notna().sum().to_string())
print("\n=== Mobile phone coverage ===")
print(cov_mobile.notna().sum().to_string())

missing_mobile = cov_mobile[MAIN_ROUNDS].isna().any(axis=1)
print(f"\nStates with any missing mobile ({MAIN_ROUNDS}):")
print(missing_mobile[missing_mobile].index.tolist())

states_all = set(panel_main["state"])
states_balanced = set(panel_est.reset_index()["state"])
print(f"\nAttrition: {len(states_all)} states in phone panel → {len(states_balanced)} with complete TFR+mobile")
print("Dropped (incomplete):", sorted(states_all - states_balanced))

def kw_match(text, keywords):
    t = str(text).lower()
    return any(k in t for k in keywords)

unmapped_phone = df_nfhs[
    (df_nfhs["area"] == "Total")
    & df_nfhs["indicator"].apply(lambda x: kw_match(x, PHONE_KW))
    & df_nfhs["harmonized_indicator"].isna()
]
print(f"\nUnmapped phone-keyword indicators: {unmapped_phone['indicator'].nunique()} unique strings")
for ind in unmapped_phone["indicator"].unique()[:10]:
    rounds = unmapped_phone.loc[unmapped_phone["indicator"] == ind, "round"].unique()
    print(f"  • {ind}  [{', '.join(rounds)}]")

flagged = df_nfhs[
    (df_nfhs["area"] == "Total")
    & df_nfhs["state"].isin(states_balanced)
    & df_nfhs["round"].isin(MAIN_ROUNDS)
    & df_nfhs["data_flag"].notna()
]
print(f"\nLong-format rows with data_flag in estimation states: {len(flagged)}")

## Phase 2 — Visualizations

In [ ]:
plot_df = panel_est.reset_index()
india_ts = df_nfhs[
    (df_nfhs["state"] == "India")
    & (df_nfhs["area"] == "Total")
    & (df_nfhs["round"].isin(MAIN_ROUNDS))
    & (df_nfhs["harmonized_indicator"].isin([INDICATORS["tfr"], INDICATORS["mobile"]]))
].pivot(index="round", columns="harmonized_indicator", values="value")
india_ts.index = india_ts.index.map(WAVE_MAP)

highlight_states = ["Kerala", "Bihar", "Uttar Pradesh", "Tamil Nadu", "Rajasthan"]

# --- Fig 1: Parallel trajectories (India + selected states) ---
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

for state, grp in plot_df[plot_df["state"].isin(highlight_states)].groupby("state"):
    axes[0].plot(grp["wave"], grp["tfr"], marker="o", ms=4, label=state)
axes[0].plot(india_ts.index, india_ts[INDICATORS["tfr"]], "k--", lw=2, label="India")
axes[0].set_xlabel("NFHS wave")
axes[0].set_ylabel("TFR (children per woman)")
axes[0].set_title("Fig 1a — Fertility trajectories")
axes[0].xaxis.set_major_locator(MaxNLocator(integer=True))
axes[0].legend(fontsize=7, loc="best")

for state, grp in plot_df[plot_df["state"].isin(highlight_states)].groupby("state"):
    axes[1].plot(grp["wave"], grp["mobile"], marker="o", ms=4, label=state)
axes[1].plot(india_ts.index, india_ts[INDICATORS["mobile"]], "k--", lw=2, label="India")
axes[1].set_xlabel("NFHS wave")
axes[1].set_ylabel("Women with own mobile phone (%)")
axes[1].set_title("Fig 1b — Mobile penetration trajectories")
axes[1].xaxis.set_major_locator(MaxNLocator(integer=True))
axes[1].legend(fontsize=7, loc="best")

fig.tight_layout()
fig.savefig(OUT_DIR / "fig1_trajectories.png", bbox_inches="tight")
plt.show()

# --- Fig 2: Cross-sectional scatter by wave ---
fig, axes = plt.subplots(1, 3, figsize=(11, 3.5), sharey=True)
for ax, (rnd, w) in zip(axes, [(r, WAVE_MAP[r]) for r in MAIN_ROUNDS]):
    sub = plot_df[plot_df["round"] == rnd]
    ax.scatter(sub["mobile"], sub["tfr"], alpha=0.75, s=35, color="#2166ac", edgecolors="white", lw=0.4)
    if len(sub) > 2:
        m, b = np.polyfit(sub["mobile"], sub["tfr"], 1)
        xline = np.linspace(sub["mobile"].min(), sub["mobile"].max(), 50)
        ax.plot(xline, m * xline + b, color="#b2182b", lw=1.5)
    ax.set_xlabel("Mobile phone (%)")
    ax.set_title(rnd)
axes[0].set_ylabel("TFR")
fig.suptitle("Fig 2 — TFR vs mobile penetration (state cross-sections)", y=1.02)
fig.tight_layout()
fig.savefig(OUT_DIR / "fig2_scatter_by_wave.png", bbox_inches="tight")
plt.show()

# --- Fig 3: Change-on-change (NFHS-4 → NFHS-6) ---
wide_chg = plot_df.pivot(index="state", columns="round", values=["tfr", "mobile"])
chg = pd.DataFrame({
    "d_tfr": wide_chg["tfr"]["NFHS-6"] - wide_chg["tfr"]["NFHS-4"],
    "d_mobile": wide_chg["mobile"]["NFHS-6"] - wide_chg["mobile"]["NFHS-4"],
}).dropna()

fig, ax = plt.subplots(figsize=(5, 4))
ax.axhline(0, color="grey", lw=0.8)
ax.axvline(0, color="grey", lw=0.8)
ax.scatter(chg["d_mobile"], chg["d_tfr"], alpha=0.8, s=40, color="#2166ac", edgecolors="white", lw=0.4)
if len(chg) > 2:
    m, b = np.polyfit(chg["d_mobile"], chg["d_tfr"], 1)
    xl = np.linspace(chg["d_mobile"].min(), chg["d_mobile"].max(), 50)
    ax.plot(xl, m * xl + b, color="#b2182b", lw=1.5)
ax.set_xlabel("Δ Mobile phone (pp), NFHS-4 → NFHS-6")
ax.set_ylabel("Δ TFR, NFHS-4 → NFHS-6")
ax.set_title("Fig 3 — Within-state changes")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig3_change_on_change.png", bbox_inches="tight")
plt.show()

# --- Fig 4: Choropleth (NFHS-6) ---
nfhs6 = panel_main[panel_main["round"] == "NFHS-6"].dropna(subset=["tfr", "mobile"])
gdf = gpd.read_file(GEO_FILE)
gdf = gdf.merge(nfhs6, left_on="state", right_on="state", how="left")

fig, axes = plt.subplots(1, 2, figsize=(10, 8))
gdf.plot(column="tfr", ax=axes[0], legend=True, cmap="YlOrRd", edgecolor="white", lw=0.3,
         legend_kwds={"label": "TFR", "shrink": 0.6})
axes[0].set_title("Fig 4a — TFR (NFHS-6)")
axes[0].axis("off")
gdf.plot(column="mobile", ax=axes[1], legend=True, cmap="Blues", edgecolor="white", lw=0.3,
         legend_kwds={"label": "Mobile (%)", "shrink": 0.6})
axes[1].set_title("Fig 4b — Women own mobile phone (NFHS-6)")
axes[1].axis("off")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig4_choropleth_nfhs6.png", bbox_inches="tight")
plt.show()

# --- Fig 5: Correlation heatmap ---
corr_cols = ["tfr", "mobile", "internet", "literacy", "schooling", "fp_modern",
             "unmet_fp", "electricity", "bank_account"]
corr_mat = panel_est.reset_index()[corr_cols].corr()

fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(corr_mat, annot=True, fmt=".2f", cmap="RdBu_r", center=0, ax=ax,
            square=True, linewidths=0.5, cbar_kws={"shrink": 0.8})
ax.set_title("Fig 5 — Pairwise correlations (estimation sample)")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig5_correlation_heatmap.png", bbox_inches="tight")
plt.show()

## Phase 3 — Panel regressions

Specifications (state-level, `area = Total`):

1. **Pooled OLS** with wave dummies and state-clustered SEs  
2. **Two-way FE** (state + wave FE) via `PanelOLS`, clustered by state  
3. **Robustness**: women's internet use (NFHS-5/6); first-difference (NFHS-4 → NFHS-6)

In [ ]:
CONTROLS = ["literacy", "schooling", "fp_modern", "electricity"]


def prep_regression_sample(panel_indexed, controls=CONTROLS, treat="mobile"):
    req = ["tfr", treat] + [c for c in controls if c in panel_indexed.columns]
    return panel_indexed.dropna(subset=req)


def run_pooled_ols(sample, treat="mobile", controls=CONTROLS):
    df = sample.reset_index()
    rhs = " + ".join([treat] + controls + ["C(wave)"])
    model = smf.ols(f"tfr ~ {rhs}", data=df)
    return model.fit(cov_type="cluster", cov_kwds={"groups": df["state"]})


def run_twoway_fe(sample, treat="mobile", controls=CONTROLS):
    exog = sample[controls + [treat]]
    mod = PanelOLS(sample["tfr"], exog, entity_effects=True, time_effects=True)
    return mod.fit(cov_type="clustered", cluster_entity=True)


def summarise_coef(result, var, label=None):
    label = label or var
    b = result.params[var]
    se = result.std_errors[var] if hasattr(result, "std_errors") else result.bse[var]
    p = result.pvalues[var]
    ci_lo, ci_hi = b - 1.96 * se, b + 1.96 * se
    stars = "***" if p < 0.01 else "**" if p < 0.05 else "*" if p < 0.10 else ""
    print(f"{label:30s}  {b:8.4f}{stars}  (SE={se:.4f}, p={p:.3f})  95% CI [{ci_lo:.4f}, {ci_hi:.4f}]")
    print(f"  → 10 pp ↑ in {label}: ΔTFR ≈ {10*b:+.3f} children per woman")
    return {"coef": b, "se": se, "p": p}


reg_sample = prep_regression_sample(panel_est)
print(f"Regression sample: N={len(reg_sample)}, states={reg_sample.index.get_level_values(0).nunique()}, waves={reg_sample.index.get_level_values(1).nunique()}\n")

print("=" * 70)
print("Model 1 — Pooled OLS (wave FE, state-clustered SEs)")
print("=" * 70)
ols_main = run_pooled_ols(reg_sample)
print(ols_main.summary().tables[1])
ols_mobile = summarise_coef(ols_main, "mobile", "Mobile phone (%)")
print(f"R² = {ols_main.rsquared:.3f}\n")

print("=" * 70)
print("Model 2 — State + Wave Fixed Effects (clustered by state)")
print("=" * 70)
fe_main = run_twoway_fe(reg_sample)
print(fe_main.summary.tables[1])
fe_mobile = summarise_coef(fe_main, "mobile", "Mobile phone (%)")
print(f"Within R² = {fe_main.rsquared_within:.3f}\n")

reg_inet = prep_regression_sample(panel_internet_est, treat="internet")
print("=" * 70)
print("Model 3a — Pooled OLS: Internet (NFHS-5/6)")
print("=" * 70)
ols_inet = run_pooled_ols(reg_inet, treat="internet")
inet_ols = summarise_coef(ols_inet, "internet", "Internet use (%)")

print("\n" + "=" * 70)
print("Model 3b — Two-way FE: Internet (NFHS-5/6)")
print("=" * 70)
fe_inet = run_twoway_fe(reg_inet, treat="internet")
inet_fe = summarise_coef(fe_inet, "internet", "Internet use (%)")

fd = panel_est.reset_index()
fd_wide = fd.pivot(index="state", columns="wave")
fd_df = pd.DataFrame({
    "state": fd_wide.index,
    "d_tfr": fd_wide["tfr"][6] - fd_wide["tfr"][4],
    "d_mobile": fd_wide["mobile"][6] - fd_wide["mobile"][4],
})
for c in CONTROLS:
    fd_df[f"d_{c}"] = fd_wide[c][6] - fd_wide[c][4]
fd_df = fd_df.dropna()

print("\n" + "=" * 70)
print("Model 4 — First difference (NFHS-4 → NFHS-6, clustered by state)")
print("=" * 70)
fd_controls = " + ".join(f"d_{c}" for c in CONTROLS)
fd_mod = smf.ols(f"d_tfr ~ d_mobile + {fd_controls}", data=fd_df)
fd_res = fd_mod.fit(cov_type="cluster", cov_kwds={"groups": fd_df["state"]})
fd_mobile = summarise_coef(fd_res, "d_mobile", "Δ Mobile (pp)")

print("\n" + "=" * 70)
print("Sensitivity — Leave-one-state-out (FE mobile coefficient)")
print("=" * 70)
states = reg_sample.index.get_level_values(0).unique()
loo = []
for st in states:
    sub = reg_sample.loc[reg_sample.index.get_level_values(0) != st]
    try:
        r = run_twoway_fe(sub)
        loo.append({"dropped_state": st, "mobile_coef": r.params["mobile"], "p": r.pvalues["mobile"]})
    except Exception:
        pass
loo_df = pd.DataFrame(loo).sort_values("mobile_coef")
print(f"Min coef: {loo_df['mobile_coef'].min():.4f} ({loo_df.iloc[0]['dropped_state']})")
print(f"Max coef: {loo_df['mobile_coef'].max():.4f} ({loo_df.iloc[-1]['dropped_state']})")
print(f"Baseline FE coef: {fe_mobile['coef']:.4f}")

results = {"ols_mobile": ols_mobile, "fe_mobile": fe_mobile, "inet_ols": inet_ols, "inet_fe": inet_fe, "fd_mobile": fd_mobile}

## Phase 4 — Economic interpretation

### Pooled OLS vs state fixed effects

The pooled OLS estimate captures **between-state variation**: high-development states tend to have both greater mobile penetration and lower TFR (Kerala, Tamil Nadu vs Bihar, Uttar Pradesh). A negative OLS coefficient is consistent with that cross-sectional gradient, but it conflates technology adoption with persistent unobserved heterogeneity — cultural fertility norms, historical development, health infrastructure, and urbanisation patterns that differ across states and are slow to change.

The **state + wave FE** specification identifies β from **within-state changes** in mobile penetration over NFHS-4 → NFHS-6, net of national fertility/technology trends (wave FE). If the FE coefficient attenuates toward zero or changes sign relative to OLS, that pattern suggests **positive omitted-variable bias** in the pooled estimate: time-invariant state traits drive both variables rather than mobile adoption causing lower fertility.

### Magnitudes

Interpret coefficients as **associations**, not causal effects. A coefficient of −0.01 on mobile (percentage-point scale) implies that a 10 percentage-point increase in women's mobile ownership is associated with a 0.10 decline in TFR — economically modest at the aggregate level, but meaningful if causal (roughly 5% of India's NFHS-6 TFR of 2.0).

### Robustness: internet

Internet use (NFHS-5/6 only) provides a shorter panel but captures a broader digital-access margin. Convergence of sign and significance across mobile and internet strengthens the descriptive case; divergence suggests measurement or mechanism differences (smartphone vs any internet access).

### Limitations (must appear in any draft)

1. **Ecological inference** — state-level correlations need not hold at the individual level (ecological fallacy).
2. **No causal identification** — reverse causality is plausible (fertility decline → women's empowerment → phone access).
3. **Fact-sheet data** — published estimates without micro-level SEs; measurement error may attenuate coefficients.
4. **Short panel** — three phone waves and ~30 clusters limit power; cluster-SE asymptotics are approximate.
5. **No population weights** — states weighted equally; large states (UP, Maharashtra) not weighted by population.
6. **Missing wealth controls** — no harmonised wealth quintile; electricity and schooling are imperfect proxies.

### Suggested next empirical checks

- Mediation: mobile → modern contraceptive use → TFR (aggregate Baron-Kenny, interpret as descriptive).
- Urban vs rural subsamples (separate panels using `area`).
- Merge Census state population weights for weighted OLS.
- Re-estimate when Pakhare & Joshi release **district-level** harmonised data (~700 districts × 3 waves) with district FE.

In [ ]:
# Dynamic summary keyed to estimated coefficients
def bias_story(ols_b, fe_b):
    if ols_b < 0 and fe_b > ols_b:
        return ("OLS shows a negative cross-state gradient, but FE attenuates the estimate — "
                "consistent with positive OVB from time-invariant development traits.")
    if ols_b < 0 and fe_b < 0:
        return ("Both OLS and FE point negative; within-state co-movement aligns with the "
                "cross-sectional pattern (still not causal).")
    return "OLS and FE diverge in sign; cross-state composition effects likely dominate OLS."

print("=== Automated interpretation (this sample) ===\n")
print(bias_story(results["ols_mobile"]["coef"], results["fe_mobile"]["coef"]))
print(f"\nPooled OLS (mobile):  β = {results['ols_mobile']['coef']:.4f}, p = {results['ols_mobile']['p']:.3f}")
print(f"State+Wave FE (mobile): β = {results['fe_mobile']['coef']:.4f}, p = {results['fe_mobile']['p']:.3f}")
print(f"FE internet robustness: β = {results['inet_fe']['coef']:.4f}, p = {results['inet_fe']['p']:.3f}")
print(f"First difference (Δ mobile): β = {results['fd_mobile']['coef']:.4f}, p = {results['fd_mobile']['p']:.3f}")
print(f"\nOutputs saved to: {OUT_DIR.resolve()}")